# Data Acquisition Project - Smithsonian Holocene Volcano Eruptions List

### Introduction

<font size="3">This project focuses on collecting, combining and analysing data about volcanoes around the world. The aim is to use an existing Kaggle dataset and additional information from the Smithsonian Global Volcanism Program website.</font>

<font size="3">Python libraries such as pandas and BeautifulSoup are used to extract, clean and merge the datasets. The merged data will then be explored using Seaborn visualisations to identify patterns and relationships between different volcano characteristics.</font>

<font size="4">Installing the required Python libraries for data processing, web scraping, Excel files, visualisation, and Jupyter Notebook.</font>

In [9]:
%pip install pandas beautifulsoup4 openpyxl seaborn matplotlib ipykernel

Note: you may need to restart the kernel to use updated packages.


<font size="4">Display the current Python version, file path of the Python environment being used and that the libraries have been imported successfully.</font>

In [10]:
import sys
import pandas as pd
from bs4 import BeautifulSoup

print("Python version:", sys.version)
print("Environment:", sys.executable)
print("Everything is working!")

Python version: 3.13.3 (tags/v3.13.3:6280bb5, Apr  8 2025, 14:47:33) [MSC v.1943 64 bit (AMD64)]
Environment: c:\Users\nimmy\OneDrive - Technological University Dublin\3rd Year\VS Code\Python\volcanos\.venv\Scripts\python.exe
Everything is working!


<font size="4">Read the saved HTML file and use BeautifulSoup to locate the volcano table and extract its rows.</font>

<font size="3">I first used Chrome Developer Tools to inspect the Smithsonian volcano website and find the HTML table containing the data. I tried using Python's urllib.request to access the website, but it returned an HTTP 403 Forbidden error because of Cloudflare security.</font>

<font size="3">To solve this, I opened the website in Chrome and saved the webpage as an HTML file (volcano.html). I then used BeautifulSoup in Python to read the saved file and extract the volcano information. The data was stored in a pandas DataFrame and saved as a CSV file.</font>

<font size="3">Python did not access the live website directly. Instead, it extracted the data from the HTML file that I had saved using Chrome.</font>

In [2]:
with open("volcano.html", "rb") as file:
    raw_html = file.read()

soup = BeautifulSoup(raw_html, "html.parser")

volcano_div = soup.find("div", class_="VolcanoListTable")

if volcano_div is None:
    raise ValueError("Volcano table not found in the saved HTML")

volcano_table = volcano_div.find("table")

rows = volcano_table.select("tbody tr")

print("Rows found:", len(rows))

Rows found: 1214


<font size="4">Extract each volcano's unique number, name, country, region, landform, and activity evidence from the HTML table.</font>

In [3]:
from urllib.parse import urlparse, parse_qs

volcano_data = []

for row in rows:
    columns = row.find_all("td")

    if len(columns) >= 5:
        link = columns[0].find("a", href=True)

        volcano_number = None

        if link:
            volcano_number = parse_qs(
                urlparse(link["href"]).query
            ).get("vn", [None])[0]

        volcano_data.append({
            "Number": volcano_number,
            "Name": columns[0].get_text(strip=True),
            "Country": columns[1].get_text(strip=True),
            "Volcanic Region": columns[2].get_text(strip=True),
            "Volcano Landform": columns[3].get_text(strip=True),
            "Evidence": columns[4].get_text(strip=True)
        })

<font size="4">Convert the scraped volcano data into a DataFrame and check its structure, first five rows, and missing values.</font>

In [4]:
scraped_df = pd.DataFrame(volcano_data)

display(scraped_df.head())

print("Dataset shape:", scraped_df.shape)
print("Missing values:")
print(scraped_df.isnull().sum())

,Number,Name,Country,Volcanic Region,Volcano Landform,Evidence
0,283001,Abu,Japan,Nankai Volcanic Arc,Cluster,Eruption Dated
1,355096,Acamarachi,Chile,Central Andean Volcanic Arc,Composite,Evidence Credible
2,342080,Acatenango,Guatemala,Central America Volcanic Arc,Composite,Eruption Observed
3,321040,Adams,United States,High Cascades Volcanic Arc,Composite,Eruption Dated
4,333050,Adams Seamount,United Kingdom,Pitcairn Hotspot Volcano Group,Minor,Eruption Dated


Dataset shape: (1214, 6)
Missing values:
Number              0
Name                0
Country             0
Volcanic Region     0
Volcano Landform    0
Evidence            0
dtype: int64


<font size="4">Save the scraped volcano data as a CSV file without including the DataFrame index.</font>

In [5]:
scraped_df.to_csv("scraped_volcanoes.csv", index=False)

<font size="4">Load the Kaggle Excel dataset into a DataFrame, display its shape and first five rows.</font>

<font size="3">For this project, I chose a dataset about volcanoes from Kaggle. The dataset contains **1,508 volcano records and 12 columns**, with information about volcanoes around the world.</font>

<font size="3">The dataset includes details such as volcano name, country, region, volcano type, activity evidence, last known eruption, latitude, longitude, elevation, dominant rock type and tectonic setting. Each volcano also has a unique identification number.</font>

<font size="3">I imported the dataset into Python using pandas and explored its structure by checking the column names, data types, number of rows and columns, and missing values. I also displayed the first five rows to better understand the data.</font>

<font size="3">I chose this dataset because I was interested in exploring different types of volcanoes and their characteristics. It could also be combined with information from the Smithsonian Global Volcanism Program website using the unique volcano identification number.</font>

In [6]:
kaggle_df = pd.read_excel("volcanos.xlsx")

print("Kaggle shape:", kaggle_df.shape)

display(kaggle_df.head())

Kaggle shape: (1508, 12)


,Number,Name,Country,Region,Type,Activity Evidence,Last Known Eruption,Latitude,Longitude,Elevation (Meters),Dominant Rock Type,Tectonic Setting
0,210010,West Eifel Volcanic Field,Germany,Mediterranean and Western Asia,Maar(s),Eruption Dated,8300 BCE,50.170,6.85,600,Foidite,Rift Zone / Continental Crust (>25 km)
1,210020,Chaine des Puys,France,Mediterranean and Western Asia,Lava dome(s),Eruption Dated,4040 BCE,45.775,2.97,1464,Basalt / Picro-Basalt,Rift Zone / Continental Crust (>25 km)
2,210030,Olot Volcanic Field,Spain,Mediterranean and Western Asia,Pyroclastic cone(s),Evidence Credible,Unknown,42.170,2.53,893,Trachybasalt / Tephrite Basanite,Intraplate / Continental Crust (>25 km)
3,210040,Calatrava Volcanic Field,Spain,Mediterranean and Western Asia,Pyroclastic cone(s),Eruption Dated,3600 BCE,38.870,-4.02,1117,Basalt / Picro-Basalt,Intraplate / Continental Crust (>25 km)
4,211001,Larderello,Italy,Mediterranean and Western Asia,Explosion crater(s),Eruption Observed,1282 CE,43.250,10.87,500,No Data,Subduction Zone / Continental Crust (>25 km)


<font size="4">Prepare the volcano numbers and merge the Kaggle dataset with the scraped website data using a left join.</font>

In [7]:
kaggle_df["Number"] = pd.to_numeric(
    kaggle_df["Number"], errors="coerce"
)

scraped_df["Number"] = pd.to_numeric(
    scraped_df["Number"], errors="coerce"
)

website_columns = scraped_df[
    ["Number", "Volcanic Region", "Volcano Landform", "Evidence"]
].dropna(subset=["Number"])

merged_df = kaggle_df.merge(
    website_columns,
    on="Number",
    how="left",
    indicator=True,
    validate="one_to_one"
)

display(merged_df.head())

,Number,Name,Country,Region,Type,Activity Evidence,Last Known Eruption,Latitude,Longitude,Elevation (Meters),Dominant Rock Type,Tectonic Setting,Volcanic Region,Volcano Landform,Evidence,_merge
0,210010,West Eifel Volcanic Field,Germany,Mediterranean and Western Asia,Maar(s),Eruption Dated,8300 BCE,50.170,6.85,600,Foidite,Rift Zone / Continental Crust (>25 km),Central European Volcanic Province,Cluster,Eruption Dated,both
1,210020,Chaine des Puys,France,Mediterranean and Western Asia,Lava dome(s),Eruption Dated,4040 BCE,45.775,2.97,1464,Basalt / Picro-Basalt,Rift Zone / Continental Crust (>25 km),Western European Volcanic Province,Cluster,Eruption Dated,both
2,210030,Olot Volcanic Field,Spain,Mediterranean and Western Asia,Pyroclastic cone(s),Evidence Credible,Unknown,42.170,2.53,893,Trachybasalt / Tephrite Basanite,Intraplate / Continental Crust (>25 km),Western European Volcanic Province,Cluster,Evidence Credible,both
3,210040,Calatrava Volcanic Field,Spain,Mediterranean and Western Asia,Pyroclastic cone(s),Eruption Dated,3600 BCE,38.870,-4.02,1117,Basalt / Picro-Basalt,Intraplate / Continental Crust (>25 km),Western European Volcanic Province,Cluster,Eruption Dated,both
4,211001,Larderello,Italy,Mediterranean and Western Asia,Explosion crater(s),Eruption Observed,1282 CE,43.250,10.87,500,No Data,Subduction Zone / Continental Crust (>25 km),NaN,NaN,NaN,left_only


<font size="4">Count the successfully matched volcano records and calculate the percentage of Kaggle data matched with the scraped website data.</font>

In [8]:
matched = (merged_df["_merge"] == "both").sum()

match_percentage = matched / len(kaggle_df) * 100

print("Kaggle records:", len(kaggle_df))
print("Website records:", len(scraped_df))
print("Matched records:", matched)
print(f"Match percentage: {match_percentage:.2f}%")

Kaggle records: 1508
Website records: 1214
Matched records: 1199
Match percentage: 79.51%
